# Temperature steering — two parallel manifolds (numeric + H2O)

Range **−20 → 110 °C** (captures steam, not just boiling). Two encodings of the *same* physical scale plus an orthogonal substance probe:

**A. Numeric proforma (number-swap, like probability).** A pool of instrument-reading questions (Gemini, once) + a fixed answer frame with a `{t}` slot. Every vector = swap the number.
- `temp_c5`  — °C, 5° steps, −20→110 = **27** steer vectors.
- `temp_f5`  — °F = the c5 grid converted (5°C→69°F → all integers), **27** vectors, *matched physical twins* of c5 → cross-unit alignment probe.
- `temp_c1`  — °C, 1° steps, −20→110 = **131** steer vectors, split into **4 chunks** (each infer <3h) + recombine.

**The unavoidable plot.** Carriers are identical across all levels (only `{t}` changes), so each trained vector is `v_T = mean_pos(T) − baseline` and **`v_{T+1}−v_T` is the pure one-degree-of-temperature direction**. Plot **‖v_{T+1}−v_T‖ vs T** → if H1 holds it spikes at the phase transitions **0°C and 100°C** and is flat in the boring middle. At 5° those peaks are invisible; at 1° they're unmissable. (overlay the 5° sampling to show *why* coarse misses it.)

**B. Rich H2O (NOT a proforma).** `h2o_v1` — 18 substance-identity vectors vs a shared neutral baseline. We sample H2O so **temperature** and **substance-identity** are two partly-covarying axes:
- frozen cluster (<0°C): snow, ice, hail, frost, sleet, slush — should clump on the temp axis, stay distinct in identity (H1+H2 in one figure).
- liquid (mild): rain, dew, fog, liquid water.  gas (hot): steam.
- adjective-water row (fixed substance, varying temp, in *words*): icy / cold / cool / room-temp / warm / hot / boiling water.
- cross-check: does “icy water” land near “ice/snow”, “boiling water” near “steam”?

Kernel: `steering`.

In [ ]:
import sys, os, asyncio, re, math
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display
REPO = Path(".")
sys.path.insert(0, str(REPO / "new_pipeline"))
import pipeline as P     # parse_numbered / _classify_prompt / _trigrams / _jaccard

def load_api_key():
    k = os.environ.get("GEMINI_API_KEY")
    if k: return k.strip()
    p = Path.home() / ".api" / "gemini_api_key"
    if p.exists(): return p.read_text().strip()
    raise SystemExit("No Gemini key")

GEN_MODEL = "gemini-3.5-flash"
print("ready | model:", GEN_MODEL, "| key:", "ok" if load_api_key() else "MISSING")

In [ ]:
from google import genai
from google.genai import types
MAX_CALLS = 60
_call_count = 0
async def gemini(prompts, *, temperature=1.0, max_output_tokens=1400, concurrency=8, thinking_budget=0):
    global _call_count
    if _call_count + len(prompts) > MAX_CALLS:
        raise RuntimeError(f"would exceed MAX_CALLS={MAX_CALLS}: {_call_count}+{len(prompts)}. reset_calls() or raise it.")
    client = genai.Client(api_key=load_api_key())
    cfg = types.GenerateContentConfig(temperature=temperature, max_output_tokens=max_output_tokens,
        thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget))
    sem = asyncio.Semaphore(concurrency); out = [""] * len(prompts)
    async def one(i, pr):
        async with sem:
            for a in range(4):
                try:
                    r = await client.aio.models.generate_content(model=GEN_MODEL, contents=pr, config=cfg)
                    cand = (r.candidates or [None])[0]
                    parts = (cand.content.parts if (cand and cand.content and cand.content.parts) else []) or []
                    out[i] = "".join(pt.text for pt in parts if getattr(pt, "text", None)); return
                except Exception as e:
                    if a == 3: out[i] = f"[ERROR: {e}]"
                    else: await asyncio.sleep(1.5 * (a + 1))
    await asyncio.gather(*(one(i, p) for i, p in enumerate(prompts)))
    _call_count += len(prompts); print(f"[gemini] +{len(prompts)} calls ({_call_count}/{MAX_CALLS})")
    return out
def reset_calls():
    global _call_count; _call_count = 0; print("reset")

## SECTION A — numeric proforma (°C / °F)
### KNOB 1 — levels + °F conversion + 1° chunking

In [ ]:
C5 = list(range(-20, 111, 5))         # -20,-15,...,110  (27 levels)
C1 = list(range(-20, 111, 1))         # -20,...,110      (131 levels)
def c_to_f(c): return c * 9 // 5 + 32  # exact integer for multiples of 5 (5°C step -> 9°F step)
F5 = [c_to_f(c) for c in C5]          # matched physical twins of C5 (all integers)

def chunkify(lst, k):
    n = math.ceil(len(lst) / k)
    return [lst[i:i+n] for i in range(0, len(lst), n)]
C1_CHUNKS = chunkify(C1, 4)           # 4 contiguous chunks ~33 each -> separate SLURM datasets temp_c1a..d

def tslug(t): return (f"neg{abs(t)}" if t < 0 else str(t))   # filesafe vector_id token

print(f"C5: {len(C5)} levels {C5[0]}..{C5[-1]}")
print(f"F5: {F5[0]}..{F5[-1]}°F  (e.g. 0°C={c_to_f(0)}°F, 37°C={c_to_f(35)}°F-ish, 100°C={c_to_f(100)}°F, 110°C={c_to_f(110)}°F)")
print(f"C1: {len(C1)} levels -> {len(C1_CHUNKS)} chunks of sizes {[len(c) for c in C1_CHUNKS]} (ranges "
      + ", ".join(f'{c[0]}..{c[-1]}' for c in C1_CHUNKS) + ")")

### KNOB 2 — proforma templates  *(instrument readings: range-robust for −20→110; only `{t}` changes)*

In [ ]:
# Instrument-reading questions: any value -20..110 is coherent (thermometer/gauge/probe can read anything),
# so every (prompt, t) pair is sensible and carriers stay IDENTICAL across all levels (needed for the curvature plot).
TEMP_PROMPTGEN = """Write {n} short, varied questions a user might ask an AI assistant to read off or state the current temperature shown on an instrument (a thermometer, gauge, sensor, probe, thermostat, or digital display) in a plausible everyday, kitchen, lab, or outdoor setting.
Each names the setting briefly but does NOT state the temperature value or hint whether it is hot or cold. One sentence each, self-contained (no files / images / links), nothing that presumes a previous conversation, history or task. Do NOT state any temperature yourself.
Return \"1.\" ... \"{n}.\" only."""

# answer frames that STATE a temperature — rotated across prompts for phrasing diversity. {t} is the only slot.
BODY_C = [
    "The thermometer reads about {t}°C.",
    "It's showing roughly {t}°C right now.",
    "The reading comes to {t}°C.",
    "Right now it's sitting at {t}°C.",
    "The display has settled at {t}°C.",
]
BODY_F = [s.replace("°C", "°F") for s in BODY_C]
# answer frames giving NO figure (steer-negative).
ABSENT = [
    "The instrument isn't giving a steady reading right now — hard to call a number.",
    "I can't make out a specific figure from the readout at the moment.",
    "There's no clear value on the display just now.",
]

def render_promptgen(n): return TEMP_PROMPTGEN.format(n=n)
def body_c(i, t):        return BODY_C[i % len(BODY_C)].format(t=t)
def body_f(i, t):        return BODY_F[i % len(BODY_F)].format(t=t)
def render_absent(i):    return ABSENT[i % len(ABSENT)]

def filter_prompts(cands, n, th=0.8):
    kept, grams = [], []
    for p in cands:
        if len(kept) >= n: break
        if P._classify_prompt(p)[0] == "hard": continue
        g = P._trigrams(p)
        if max((P._jaccard(g, kg) for kg in grams), default=0) >= th: continue
        kept.append(p); grams.append(g)
    return kept
print("proforma templates loaded")

### TOY A — the ladder at the extremes (1 Gemini call for real prompts, then pure string-render)
Eyeball: do the frames read naturally at −20, 0, 37, 100, 110°C, and do the °F twins match the same physical points?

In [ ]:
_raw = (await gemini([render_promptgen(10)], temperature=1.0, max_output_tokens=900))[0]
toy_prompts = filter_prompts(P.parse_numbered(_raw, 10), 6)
print("PROMPTS (instrument-reading questions):")
for i, p in enumerate(toy_prompts): print(f"  {i}. {p}")

SAMPLE_C = [-20, 0, 35, 100, 110]
out = ["### °C ladder (pos = states t°C, neg = absent)"]
for t in SAMPLE_C:
    i = SAMPLE_C.index(t)
    out.append(f"- **{t}°C** — _q:_ {toy_prompts[i]}\n    - **pos:** {body_c(i, t)}\n    - **neg:** {render_absent(i)}")
out.append("\n### °F twins (same physical points, matched carriers)")
for t in SAMPLE_C:
    i = SAMPLE_C.index(t)
    out.append(f"- **{t}°C ≡ {c_to_f(t) if t%5==0 else round(t*9/5+32)}°F** — **pos:** {body_f(i, c_to_f(t) if t%5==0 else round(t*9/5+32))}")
display(Markdown("\n".join(out)))

### Vector-count summary (Section A)

In [ ]:
print(f"temp_c5 : {len(C5)} steer vectors")
print(f"temp_f5 : {len(F5)} steer vectors (matched twins of c5)")
print(f"temp_c1 : {len(C1)} steer vectors -> {len(C1_CHUNKS)} datasets temp_c1a..{chr(96+len(C1_CHUNKS))} "
      f"(sizes {[len(c) for c in C1_CHUNKS]})")
print(f"TOTAL numeric: {len(C5)+len(F5)+len(C1)} vectors")

## SECTION B — rich H2O manifold (not a proforma)
### KNOB 3 — the 18 H2O members (with nominal temperature for the geometry projection)

In [ ]:
H2O = [
    # frozen cluster (<0°C) — should clump on the temp axis but stay distinct in identity
    dict(slug="snow",   label="snow",                  desc="falling or fallen snow",         temp_c=-5,  group="frozen"),
    dict(slug="ice",    label="ice",                   desc="solid ice",                       temp_c=-5,  group="frozen"),
    dict(slug="hail",   label="hail",                  desc="hail / hailstones",               temp_c=-5,  group="frozen"),
    dict(slug="frost",  label="frost",                 desc="frost on surfaces",               temp_c=-3,  group="frozen"),
    dict(slug="sleet",  label="sleet",                 desc="sleet (rain mixed with ice)",     temp_c=0,   group="frozen"),
    dict(slug="slush",  label="slush",                 desc="slush (partly melted snow)",      temp_c=0,   group="frozen"),
    # liquid forms (mild)
    dict(slug="rain",   label="rain",                  desc="rain / rainfall",                 temp_c=12,  group="liquid"),
    dict(slug="dew",    label="dew",                   desc="dew on grass or leaves",          temp_c=12,  group="liquid"),
    dict(slug="fog",    label="fog",                   desc="fog or mist",                     temp_c=12,  group="liquid"),
    dict(slug="puddle", label="liquid water",          desc="a puddle of liquid water",        temp_c=15,  group="liquid"),
    # gas (hot)
    dict(slug="steam",  label="steam",                 desc="steam / water vapour rising",     temp_c=100, group="gas"),
    # adjective-qualified liquid water — a temperature sweep on a FIXED substance, in words (not numbers)
    dict(slug="icy_water",      label="icy water",              desc="icy water",              temp_c=1,   group="adj"),
    dict(slug="cold_water",     label="cold water",             desc="cold water",             temp_c=7,   group="adj"),
    dict(slug="cool_water",     label="cool water",             desc="cool water",             temp_c=15,  group="adj"),
    dict(slug="roomtemp_water", label="room-temperature water", desc="room-temperature water", temp_c=22,  group="adj"),
    dict(slug="warm_water",     label="warm water",             desc="warm water",             temp_c=40,  group="adj"),
    dict(slug="hot_water",      label="hot water",              desc="hot water",              temp_c=65,  group="adj"),
    dict(slug="boiling_water",  label="boiling water",          desc="boiling water",          temp_c=100, group="adj"),
]
from collections import Counter
print(f"{len(H2O)} members | groups: {dict(Counter(m['group'] for m in H2O))}")
for m in H2O: print(f"  {m['slug']:16s} {m['temp_c']:>4}°C  {m['group']:6s}  {m['label']}")

### KNOB 4 — rich PAIRED generation (frequency-style `v = (B|A) − (A|A)`, NOT pooled)
Built exactly like frequency *steer*: one shared pool of neutral scene-requests; for each request we write **two continuations of the same prompt** — a positive that features the member, and an `absent` negative mentioning no water at all. The absent negative is generated once per prompt and **shared across all 18 members (paired by prompt)**, so every member vector = `mean(pos_m) − mean(neg)` over identical prompts and the only systematic difference is the member itself. No independent pos/neg distributions. ~72 prompts × (1 neg + 18 pos) ≈ 1.4k continuation calls (same order as frequency).

In [ ]:
# H2O is built like frequency STEER: pos and neg are two continuations of the SAME neutral scene-request.
# absent neg is generated once per prompt and shared across all 18 members (paired by prompt) -> every
# member vector = mean(pos_m) - mean(neg) over identical prompts; only the member differs. NO pooled batches.

H2O_PROMPTGEN = """Write {n} short, varied requests a user might send an AI assistant asking it to describe or narrate a brief, concrete everyday scene or moment (a morning routine, a view from a window, a walk, a kitchen, a street, a garden), indoor or outdoor.
One sentence each, self-contained (no files / images / links), nothing that presumes a previous conversation, history or task, not a yes/no question. Do NOT mention water, weather, or temperature yourself.
Return "1." ... "{n}." only."""

# pos: continue the SAME request so the member features prominently and naturally.
H2O_CONT = """Write one short, natural response an AI assistant would give to the request below, describing the scene so that {desc} features prominently and naturally in it.
Weave it in concretely with a sensory detail or two (a few sentences); do not state any numeric temperature.

Request: {prompt}

Response only - no preamble."""

# neg (absent, SHARED across members): continue the SAME request with no water in any form.
H2O_CONT_ABSENT = """Write one short, natural response an AI assistant would give to the request below, describing the scene naturally.
Do NOT mention water in any form - no rain, snow, ice, frost, hail, sleet, slush, dew, fog, mist, steam, vapour, puddles, or drinking water - and do not state any temperature. A few sentences.

Request: {prompt}

Response only - no preamble."""

def render_h2o_promptgen(n):       return H2O_PROMPTGEN.format(n=n)
def render_h2o_cont(desc, prompt): return H2O_CONT.format(desc=desc, prompt=prompt)
def render_h2o_absent(prompt):     return H2O_CONT_ABSENT.format(prompt=prompt)
# held-out steering-eval prompts reuse the neutral scene-request generator (distinct batch; open-ended).
print("h2o paired-continuation templates loaded (frequency-style v=(B|A)-(A|A))")

### TOY B — the paired construction on a shared prompt (~19 Gemini calls)
For each shared scene-request you'll see the single `absent` negative and every member's positive **continuing the same prompt**. Check: is it genuinely the same prompt across pos & neg? does each member feature naturally & distinctly? is the negative truly water-free? (lower the prompt/member counts for fewer calls.)

In [ ]:
SHOW = [m for m in H2O if m["slug"] in ("snow", "ice", "hail", "steam", "icy_water", "warm_water", "boiling_water")]
_raw = (await gemini([render_h2o_promptgen(10)], temperature=1.0, max_output_tokens=900))[0]
prompts = filter_prompts(P.parse_numbered(_raw, 10), 3)            # one shared pool of neutral scene-requests
neg = await gemini([render_h2o_absent(p) for p in prompts], temperature=0.7, max_output_tokens=300)   # absent, shared across members
pos = {}
for m in SHOW:
    pos[m["slug"]] = await gemini([render_h2o_cont(m["desc"], p) for p in prompts], temperature=0.7, max_output_tokens=300)

for j, p in enumerate(prompts):
    out = [f"### shared prompt {j}: _{p}_", f"- **neg (absent | prompt):** {neg[j]}"]
    for m in SHOW:
        out.append(f"- **pos · {m['slug']} ({m['temp_c']}°C):** {pos[m['slug']][j]}")
    display(Markdown("\n".join(out))); display(Markdown("---"))
display(Markdown("_Same prompt -> paired pos/neg. Member vector = mean(pos_m) - mean(shared neg) over these prompts; the absent neg is identical across all members._"))

## Next (after you approve the toys)
1. `generate_temp_numeric_v1.py` — one shared instrument-prompt pool, paired pos(`{t}°C`)/neg(absent); emits `temp_c5`, `temp_f5`, `temp_c1a..d` (proforma; ~few API calls).
2. `generate_h2o_v1.py` — frequency-style paired continuations: shared neutral scene-prompts → per-prompt `absent` neg (shared across members) + per-member pos; held-out = a distinct neutral batch. Emits `h2o_v1` (~1.4k calls).
3. convert each → `make_configs.py` for all per-layer YAMLs → `submit_temperature_v1.sh` (c5/f5/h2o now; c1a..d as a deferred parallel batch) → `recombine_temp_c1.py`.
4. analysis: **`‖v_{T+1}−v_T‖ vs T`** curvature plot (the unavoidable figure) + the two-manifold projection (H2O members onto the temp axis).